# 🧪 W10-D1 Permission 与 Policy：策略只能收紧

> 配套阅读：同名 `.md`。本 notebook 只用小规模、可重复的模拟来验证核心治理约束。

**实验目标：** 比较角色权限、PolicyBundle 不变量与运行时“只可收紧”的判定结果。


In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

# 三层角色的静态权限矩阵：权限由控制面给出，而非模型临时决定。
roles = {
    "platform_admin": {"tenant.manage", "audit.read", "contract.read", "contract.approve"},
    "tenant_admin": {"audit.read", "contract.read", "contract.approve"},
    "member": {"contract.read"},
}
requests = [("member", "contract.read"), ("member", "contract.approve"),
            ("tenant_admin", "contract.approve"), ("platform_admin", "tenant.manage")]
for role, action in requests:
    print(f"{role:14s} {action:18s} -> {action in roles[role]}")

plt.figure(figsize=(7, 2.8))
permission_counts = [len(roles[r]) for r in roles]
plt.bar(list(roles), permission_counts, color=["#4C78A8", "#59A14F", "#E15759"])
plt.ylabel("允许的权限数"); plt.title("角色权限是预先定义的静态边界")
plt.tight_layout(); plt.show()


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class PolicyBundle:
    effect_policy: str
    review_gate_policy: str
    def __post_init__(self):
        if self.effect_policy == "conditional_write" and self.review_gate_policy == "none":
            raise ValueError("写操作必须配置审批门")

for pair in [("read_only", "none"), ("conditional_write", "mandatory"),
             ("conditional_write", "none")]:
    try:
        print(pair, "->", PolicyBundle(*pair))
    except ValueError as err:
        print(pair, "-> 拒绝：", err)

bundle = PolicyBundle("conditional_write", "mandatory")
try:
    bundle.effect_policy = "read_only"
except Exception as err:
    print("冻结后的篡改被阻止：", type(err).__name__)


In [ ]:
# Runtime 可以比策略更严格，但不能把 read_only 放宽为 write。
def authorize(bundle, action, runtime_restriction=False):
    is_write = action.endswith("approve")
    if runtime_restriction:
        return "DENY: runtime 收紧"
    if is_write and bundle.effect_policy == "read_only":
        return "DENY: policy 禁止写"
    if is_write and bundle.review_gate_policy != "none":
        return "PENDING_APPROVAL"
    return "ALLOW"

for policy in [PolicyBundle("read_only", "none"), PolicyBundle("conditional_write", "mandatory")]:
    print(policy, "读 ->", authorize(policy, "contract.read"),
          "；写 ->", authorize(policy, "contract.approve"))
print("额外收紧 ->", authorize(PolicyBundle("conditional_write", "mandatory"), "contract.read", True))
